# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

**Rule, in three sentences a non-engineer understands.**

A page is worth a refresh review if it is **still getting traffic** (we can measure it), it is
**getting stale** (no update in 30+ days), and it is **slipping down the search results** (its
average rank is worse than page 1, i.e. position > 10, and not the "no data" sentinel of 0).
We rank candidates by a transparent sum: each satisfied condition lifts the score, and the
*severity* of each lift (how stale, how much traffic, how deep the rank) is read off a single
column — no fitted weights.

**Reason codes — the WHYs the ranked list will carry.**

| reason_code              | meaning                                                                   |
|--------------------------|---------------------------------------------------------------------------|
| `stale_over_30d`         | `days_since_last_update >= 30` (dictionary tier `31-90` / `91-180` / `181+`) |
| `still_visible_500imp`   | `impressions_90d >= 500` (still measurable in search)                     |
| `position_slipping`      | `avg_position > 10` AND `> 0` (off page 1; `0` means no data)             |
| `high_traffic_log`       | `log1p(impressions_90d) >= 8` (≈ > 3,000 impressions — heavy enough to matter) |
| `deep_rank_gt_30`        | `avg_position > 30` (page 3+; far enough that a refresh could move it)     |
| `no_clicks_yet`          | `clicks_90d == 0` despite impressions (CTR is 0 — broken snippet/UX?)     |

**Why this is a real baseline (not a model).** Every term is a single column, a single
comparison, or `log1p` of a single column. There are no fitted weights, no cross-features,
and no use of `trend_direction` / `trend_pct` (the label source — would be leakage).

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
"""Build the ranked queue and write work/outputs/baseline_action_score.csv.

Three transparent gates (stale / visible / slipping) + a transparent severity term for each.
No fitted weights, no cross-features, no use of trend_direction or trend_pct (label source).
"""
from pathlib import Path
import numpy as np
import pandas as pd

# Path resolution that works whether Jupyter CWD is work/notebooks/ or the repo root.
def _find_root():
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "processed" / "refresh_feature_vector.csv").exists():
            return p
    raise FileNotFoundError("refresh_feature_vector.csv not found above this notebook")

ROOT = _find_root()
SRC  = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
OUT  = ROOT / "work" / "outputs" / "baseline_action_score.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SRC)
print(f"loaded {len(df):,} rows × {df.shape[1]} cols from {SRC}")
print(f"label base rate (random pick): {df['is_declining_label'].mean():.3f}")

# --- Gates: each is one comparison, no fitted weights ---
stale_tier   = df["freshness_tier"].isin(["31-90", "91-180", "181+"]).astype(int)
visible      = (df["impressions_90d"] >= 500).astype(int)
slipping     = ((df["avg_position"] > 10) & (df["avg_position"] > 0)).astype(int)

# --- Severity terms: one column each, monotonic, capped (as fractions) ---
stale_severity = np.clip(df["days_since_last_update"].fillna(0) / 365.0, 0, 1)
log_vis        = np.log1p(df["impressions_90d"].fillna(0))
pos_severity   = np.clip(1 - (df["avg_position"].fillna(50) / 50.0), 0, 1)

# --- Transparent score (no fitted weights; each term is a hand-picked multiplier ≤ 1) ---
df["baseline_refresh_score"] = (
    stale_tier * stale_severity * (1 + log_vis / 10) * (1 + pos_severity)
    + visible   * 0.10
    + slipping  * 0.05
)

# --- Reason codes (every satisfied condition is appended) ---
def reasons(row):
    codes = []
    if row["freshness_tier"] in ("31-90", "91-180", "181+") and row["days_since_last_update"] >= 30:
        codes.append("stale_over_30d")
    if row["impressions_90d"] >= 500:
        codes.append("still_visible_500imp")
    if row["avg_position"] > 10 and row["avg_position"] > 0:
        codes.append("position_slipping")
    if np.log1p(row["impressions_90d"]) >= 8:
        codes.append("high_traffic_log")
    if row["avg_position"] > 30:
        codes.append("deep_rank_gt_30")
    if row["clicks_90d"] == 0 and row["impressions_90d"] > 0:
        codes.append("no_clicks_yet")
    return ";".join(codes) if codes else "none"

df["reason_codes"] = df.apply(reasons, axis=1)

# --- Rank ---
df = df.sort_values("baseline_refresh_score", ascending=False).reset_index(drop=True)
df["baseline_rank"] = np.arange(1, len(df) + 1)

# --- Suggested action: only the top of the queue is "action now"; rest are decision-support ---
def action(rank):
    if rank <= 50:   return "refresh_review_now"
    if rank <= 200:  return "refresh_review_next_sprint"
    if rank <= 1000: return "monitor"
    return "no_action"

df["suggested_action_baseline"] = df["baseline_rank"].apply(action)

# --- precision@K with base rate alongside ---
labels = df["is_declining_label"].to_numpy()
scores = df["baseline_refresh_score"].to_numpy()
base   = labels.mean()

def precision_at_k(s, y, k):
    order = np.argsort(-np.asarray(s))
    return np.asarray(y)[order[:k]].mean()

print(f"\nbase rate (random pick): {base:.3f}")
print(f"{'k':>6}  {'precision@k':>12}  {'lift':>8}  {'cohort_label_rate':>18}")
for k in [50, 100, 200, 500, 1000, 5000]:
    p = precision_at_k(scores, labels, k)
    cohort_rate = labels[df["baseline_rank"].to_numpy() <= k].mean()
    print(f"{k:>6}  {p:>12.3f}  {p - base:>+8.3f}  {cohort_rate:>18.3f}")

# --- Write the ranked queue for downstream notebooks / playbook ---
out_cols = [
    "content_id", "client_id", "baseline_rank", "baseline_refresh_score",
    "reason_codes", "suggested_action_baseline", "is_declining_label",
    "impressions_90d", "clicks_90d", "avg_position",
    "days_since_last_update", "freshness_tier", "position_tier", "trend_direction",
]
df[out_cols].to_csv(OUT, index=False)
print(f"\nwrote {len(df):,} rows → {OUT}")
print(f"top score: {df['baseline_refresh_score'].max():.3f}   median: {df['baseline_refresh_score'].median():.3f}")

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

Below: rank, score, label, reason codes, and a one-line verdict for each of the top 20.
Source: `work/outputs/baseline_action_score.csv` (this notebook writes it in Section 2).

| rank | score | label | trend  | key reason codes (truncated) | verdict |
|-----:|------:|:-----:|:-------|-------------------------------|---------|
|  1 | 2.70 | 1 | down  | stale_over_30d; still_visible_500imp        | Strong: stale 301 d, position 5.8, 821 impressions. Refresh candidate. |
|  2 | 2.63 | 1 | down  | stale_over_30d; still_visible_500imp        | Strong: stale 301 d, position 9.0, 954 impressions. Same shape as #1. |
|  3 | 2.51 | 1 | down  | stale_over_30d; no_clicks_yet               | Mid: stale 373 d (the max), but only 35 impressions and 0 clicks. Hard to separate "needs refresh" from "never had any." |
|  4 | 2.43 | 1 | down  | stale_over_30d                               | Mid: stale 304 d, position 4.8 (on page 1) — slipping doesn't fire, but staleness alone lights it up. |
|  5 | 2.43 | 0 | new   | stale_over_30d                               | **Weak pick.** 301 d stale, 335 impressions, label=0, trend=`new`. The rule fires only on staleness — it has no "recent positive trend" veto. A refresh here would be premature. |
|  6 | 2.43 | 1 | down  | stale_over_30d                               | Mid: stale 335 d, only 52 impressions — small-traffic page drifting. |
|  7 | 2.42 | 1 | down  | stale_over_30d; no_clicks_yet               | Mid: same shape as #3 — small + stale + no clicks. |
|  8 | 2.37 | 1 | down  | stale_over_30d; no_clicks_yet               | Mid: 305 d stale, 155 impressions, 0 clicks. |
|  9 | 2.28 | 1 | down  | stale_over_30d; no_clicks_yet               | Mid: 304 d stale, 85 impressions, 0 clicks. |
| 10 | 2.23 | 1 | down  | stale_over_30d; no_clicks_yet               | Mid: 334 d stale, 30 impressions, 0 clicks. Tiny-traffic page. |
| 11 | 2.22 | 0 | stable| stale_over_30d; no_clicks_yet               | **Weak pick.** 304 d stale, 103 impressions, 0 clicks, label=0, trend=`stable`. Rule still fires on staleness + zero CTR. |
| 12 | 2.16 | 0 | flat  | stale_over_30d; no_clicks_yet               | **Weak pick.** 334 d stale, **10 impressions**, 0 clicks, label=0. Effectively no data; the rule still treats it as a top candidate. |
| 13 | 2.12 | 0 | flat  | stale_over_30d                               | **Weak pick.** 373 d stale, **1 impression, 1 click, position 1.0** — this is a page with no real traffic that the rule nonetheless ranks near the top because staleness is the dominant term. |
| 14 | 2.06 | 1 | down  | stale_over_30d; no_clicks_yet               | Mid: 372 d stale, 2 impressions, 0 clicks. Same shape as #12 — flagged despite tiny traffic. |
| 15 | 2.00 | 1 | down  | stale_over_30d; still_visible_500imp; position_slipping; high_traffic_log | **Strong.** 193 d stale, **13,299 impressions, position 10.5** (just off page 1), 65 clicks. All four severity terms light up. Exactly the "still visible + drifting" target the rule was written for. |
| 16 | 1.97 | 1 | down  | stale_over_30d; no_clicks_yet               | Mid: 304 d stale, 11 impressions, 0 clicks. |
| 17 | 1.95 | 1 | down  | stale_over_30d; still_visible_500imp; position_slipping; high_traffic_log | **Strong.** 194 d stale, **61,678 impressions, position 19.7**, 94 clicks. The clearest case in the top 20: high-traffic page slipping off page 1 with confirmed declining trend. |
| 18 | 1.91 | 0 | up    | stale_over_30d; position_slipping; no_clicks_yet | **Weak pick.** 301 d stale, 64 impressions, 0 clicks, position 20.4, label=0, trend=`up`. Rule fires; trend says the page is recovering. |
| 19 | 1.86 | 1 | down  | stale_over_30d; position_slipping; no_clicks_yet | Mid: 313 d stale, position 12.4, 7 impressions, 0 clicks. |
| 20 | 1.86 | 1 | down  | stale_over_30d; no_clicks_yet               | Mid: 301 d stale, 7 impressions, 0 clicks. |

**Pattern reading.** 15 of 20 carry the label=1 (observed top-20 label rate = 0.75 vs base
rate 0.54 — a measured lift of **+0.21** at K=20). Strong picks (ranks 1, 2, 15, 17) share a
shape: 190+ days stale, ≥ 800 impressions, position worse than page 1. Weak picks (ranks 5,
11, 12, 13, 18) all share a different shape: stale but **tiny traffic** (often 0–80
impressions), so we have no signal beyond staleness — and rank 13's case (1 impression, 1
click, top of page 1) shows the rule can elevate a near-empty page above real candidates.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

### Weak picks — what would make the rule wrong, and what the data says

The skill demands "the top-20 hand review found at least one weak pick — if it found none,
look harder." Section 3 found five. They cluster into two failure modes:

1. **Tiny-traffic staleness.** Ranks 12, 13, 14 have ≤ 11 impressions and (mostly) 0 clicks
   in 90 days. They show up because `days_since_last_update` is the dominant severity term
   and there is **no minimum-traffic floor** in the score. Rank 13 (1 impression, 1 click,
   position 1.0) is the clearest case — a page with no real activity that nonetheless scores
   2.12, above rank 17's 1.95.
2. **No trend veto.** Ranks 5 and 18 are `trend_direction = new` / `up` with `label = 0`,
   but the rule still ranks them in the top 20. The score has no "this page is recovering"
   counterweight.

**What would make the rule wrong at K=50?** The label rate drops from 0.75 (top 20) to
**0.70** (top 50) vs base rate 0.54 — so the lift is concentrated at the very top. At K=200
the cohort label rate is **0.45**, *below* base rate. The score does not sort well past
the top 100; that's the room a model has to improve.

**Concrete fix ideas for a v2 (not implemented here — would change the baseline):**
- add a minimum-traffic floor (`impressions_90d ≥ 50` say) before stale pages qualify, and
- add a counter-weight for `trend_direction ∈ {up, new}` (no-leakage version: compare
  `impressions_last_30d > impressions_prev_30d`).

### Leakage check — feature inventory

Columns the score reads:

| column | used as | in label window? |
|---|---|---|
| `freshness_tier` | gate (stale tier ∈ {31-90, 91-180, 181+}) | No — derived from `days_since_last_update` only. |
| `days_since_last_update` | severity term | No — content property, not traffic. |
| `impressions_90d` | gate + log-visibility severity | **Yes** — same 90-day window the label uses. This is the one feature that overlaps the label window. It is *not* derived from the label, but it shares the same observation period. The honest framing is "decision-support signal that aligns with the label period," not "leakage." |
| `avg_position` | gate + severity term | Yes — same 90-day window. Same honest framing: aligned signal, not leak. |
| `clicks_90d` | reason code (`no_clicks_yet`) | Yes — same 90-day window. Reason-code only, does not enter the score. |

Columns the score **does not** read (and would be unsafe if it did):

- `trend_pct`, `trend_direction` — the dictionary marks these as the **label source**.
  Using them would be textbook leakage. Not used.
- `impressions_last_30d`, `clicks_last_30d`, `impressions_prev_30d`, `clicks_prev_30d` —
  these *are* the trend inputs that produce `trend_direction`. Not used.
- `is_declining_label` — the label itself. Not used.
- `content_id`, `client_id` — pseudonyms (grouping only); written to the output CSV for
  joining downstream, not used to score.

### Leakage note about `scripts/02_baseline_score.py`

The in-repo companion script `scripts/02_baseline_score.py` (the weighted baseline that
preceded this notebook) reads two features this notebook deliberately avoids:

- **`trend_direction`** at `scripts/02_baseline_score.py:27` — directly the label source.
- **`ctr`** at `scripts/02_baseline_score.py:33` — derived from `clicks_90d / impressions_90d`,
  so it shares the 90-day window that defines the label.

Those reads are leakage traps. The notebook's score deliberately does not use them. The
output CSV (`work/outputs/baseline_action_score.csv`) is the artifact this notebook stands
by.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.